<a href="https://colab.research.google.com/github/SuryamaheshPenke/Gen_AI-Notes/blob/main/Class_10_Tool_Using_LLMs_Function_Calling_and_Structured_Outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Class 10 - Tool-Using LLMs: Function Calling and Structured Outputs**
### Module 1: Prompt Engineering & LLM Mastery
---

## **Agenda**

- PART 1 - Why Tools Exist: three fundamental LLM problems
- PART 2 - How Function Calling Works Mechanically
- PART 3 - The Tool Arbiter Pattern
- PART 4 - Tool Schemas and Validation
- PART 5 - Structured Outputs and JSON Mode
- Code Demos (Demo 5: instructor library for free-text extraction)
- Key Takeaways
- Quick Check

## **PART 1 - Why Tools Exist**

LLMs are next-token predictors trained on static text corpora. That single fact creates three problems that no amount of prompting can fix. Tools were invented to solve all three.

The first tool integrations appeared in research around 2022 (Toolformer, WebGPT) and became a production API feature with OpenAI's `functions` parameter in June 2023. Every major provider has since followed.

---

### Problem 1: Hallucinations

**Simple**
When you ask an LLM a factual question it is not confident about, it guesses - and the guess comes out sounding certain. It does not say "I don't know"; it says something plausible but wrong. This is not a bug in any one model. It is what happens when you train a system to always produce the most statistically likely next token.

**Technical**
Hallucination occurs because the model's objective is to maximize the likelihood of the next token given context. There is no separate "factual verification" step. When the true answer is absent or uncertain in the training distribution, the model interpolates from patterns that superficially resemble the correct answer. Tools solve this by externalizing factual lookup - instead of generating an answer, the model generates a query to a trusted data source.

---

### Problem 2: Stale Knowledge

**Simple**
An LLM's knowledge is frozen at its training cutoff. If you ask it about something that happened after that date - a product release, a news event, a stock price - it cannot know. Retraining is expensive and slow.

**Technical**
The model's weights encode a static compression of the training corpus. Knowledge cutoff is a hard architectural constraint, not a tunable parameter. Retrieval tools (web search, database queries, RAG pipelines) solve this by allowing the model to fetch fresh data at inference time rather than relying on what was baked into the weights.

---

### Problem 3: No Math

**Simple**
LLMs are notoriously unreliable at arithmetic, unit conversions, and anything requiring precise numeric computation. Ask an LLM to multiply large numbers or calculate compound interest and it will often get it wrong. This is not fixable with prompting.

**Technical**
The model encodes numerical relationships as floating-point embeddings, not as computational logic. Arithmetic accuracy degrades as number size and operation complexity increase. The fix is to give the model a calculator tool - it generates the expression, a real interpreter evaluates it, and the result comes back exact.

## **PART 2 - How Function Calling Works Mechanically**

Function calling is not the model running code. That is the most common misconception. The model never executes anything. It outputs text in a structured format, your application executes the tool, and the result goes back to the model as a new message.

Here is the exact sequence:

1. You define available tools in your API request (name, description, parameter schema)
2. Model receives the user message plus the tool definitions
3. Model decides to call a tool and outputs a structured JSON object (name + arguments) instead of a normal reply
4. Your application parses that JSON, validates it, and executes the real function
5. You send the function result back as a new message with role `tool`
6. Model generates its final response using the real result

---

### The Model's Role

**Simple**
The model is a decision-maker and translator, not an executor. It reads the conversation, decides which tool to use and with what inputs, and writes down its request in a structured format. Your code does the actual work.

**Technical**
The model is fine-tuned to recognize when tool use is appropriate and to emit a `tool_calls` object instead of (or in addition to) a content string. The output is a JSON payload conforming to the schema you provided. The model does not have runtime access to any external system. It has learned patterns for matching user intent to tool signatures from RLHF training on tool-use demonstrations.

---

### The Application's Role

**Simple**
Your application is the actual connector to the real world. It reads the model's structured request, calls the real function (a database, an API, a calculator), and feeds the result back into the conversation.

**Technical**
The function dispatch loop is entirely in application code. You maintain a registry mapping tool names to callables. On receiving a `finish_reason: tool_calls`, you extract each `tool_call`, deserialize the `arguments` JSON, call the matching function, and append a `role: tool` message with the `tool_call_id` and the serialized result. The model then receives the updated context and produces a final response.

## **PART 3 - The Tool Arbiter Pattern**

This is the most important architectural pattern in tool-using LLM systems. Understand this and you understand how production tool use is actually built.

The problem: the model proposes a tool call. That proposal could be wrong - wrong tool, wrong parameters, malformed input, a request for a privileged action it should not have. If you execute blindly, you get bugs, security issues, or data corruption.

The Tool Arbiter sits between the model's proposal and actual execution. It validates, and either approves or rejects.

---

### The Pattern

**Simple**
Imagine a junior employee who can look things up in a database. Before they touch the database, a senior engineer reviews their query - checking it makes sense, uses the right table, and does not accidentally delete anything. The junior employee (LLM) proposes; the senior engineer (arbiter) validates; then the query runs. The model never has direct access to the real system.

**Technical**
The Tool Arbiter is a middleware component between the LLM output and tool dispatch. It receives the raw `tool_calls` payload and performs:
1. **Schema validation** - arguments match the declared types and constraints
2. **Semantic validation** - the call is coherent given the conversation context
3. **Authorization check** - the caller has permission for this operation
4. **Rate/quota enforcement** - prevent runaway loops or cost overruns
5. **Logging and audit** - every tool call is recorded before execution

If validation passes, the arbiter dispatches. If not, it returns a structured error to the model, which can then correct its call or respond to the user. The model never directly touches any external system.

---

### Why This Matters

**Simple**
Without an arbiter, a jailbroken or confused model can trigger real actions - send emails, delete records, make API calls with your credentials. The arbiter is your safety layer.

**Technical**
Prompt injection attacks can cause the model to call tools with attacker-supplied arguments. Without arbiter-level validation, this is a direct code execution vulnerability routed through your application. The arbiter enforces a separation of concerns: the model is responsible for intent, the arbiter is responsible for safety, and the tool is responsible for execution. This also enables observable, testable tool behavior - you can unit-test the arbiter independently of the model.

## **PART 4 - Tool Schemas and Validation**

A tool schema is how you communicate capability and contract to the model. A well-written schema reduces errors. A vague schema produces hallucinated arguments.

---

### Writing Tool Schemas

**Simple**
A tool schema is a description of a function that the model can read. It tells the model what the function does, what inputs it takes, and what those inputs mean. The model uses this description to decide when to call it and what arguments to pass. The clearer your description, the more accurately the model will call your tool.

**Technical**
Tool schemas follow JSON Schema format. Each schema includes a `name` (unique identifier), `description` (natural language, used by the model for routing decisions), and `parameters` (a JSON Schema object defining types, required fields, enums, descriptions, and constraints). The model was trained on examples of matching natural language requests to schemas - so the `description` and parameter `description` fields are the primary signal for correct dispatch. Keep schemas narrow: only expose parameters the model should control, hard-code everything else in your application.

---

### Pre- and Post-Validation

**Simple**
Pre-validation checks the model's request before running the tool. Post-validation checks the tool's output before sending it back. Both catch different classes of problems.

**Technical**
- **Pre-validation (arbiter):** Deserialize the `arguments` JSON, validate against the schema using a library like `jsonschema` or Pydantic, check authorization, log the attempt. Reject malformed calls with a structured error.
- **Post-validation (result handling):** Verify the tool returned a sensible result - not an error, not an empty set when a result was expected, within expected size bounds. Truncate or summarize long results before returning to the model to avoid context overflow.

Both steps prevent the model from propagating errors silently through a multi-step chain.

## **PART 5 - Structured Outputs and JSON Mode**

Function calling is one use of structured output: the model emits a structured tool call. But you can also force the model's final reply to be structured JSON - useful for extraction, classification, scoring, and any case where downstream code needs to parse the model's output.

---

### JSON Mode and `response_format`

**Simple**
Normally the model writes prose. JSON mode tells it: write valid JSON only. No prose, no markdown wrapper, just a JSON object. This is useful when your application needs to parse the model's reply as data, not read it as text.

**Technical**
OpenAI's `response_format: {"type": "json_object"}` and Anthropic's `betas: ["json_mode"]` both constrain the model's output distribution to always produce valid JSON. The model is not forced into a specific schema - it still decides the keys. For strict schema adherence, you need either OpenAI's Structured Outputs (`response_format: {"type": "json_schema", "json_schema": {...}}`) or a library like `instructor`.

---

### Pydantic Models for Schema Definition

**Simple**
Instead of writing JSON Schema by hand, you define a Python class that describes the shape of the output you want. The library converts that class to a schema, sends it to the model, and parses the response back into a typed Python object. You get type safety and validation without writing JSON Schema manually.

**Technical**
Pydantic v2 models generate JSON Schema via `.model_json_schema()`. Libraries like `instructor` wrap the OpenAI client, inject the schema into the request, and call `model.model_validate(response_json)` on the result - raising `ValidationError` if the model's output does not conform. This gives you runtime-validated, typed LLM outputs. Use `Field(description=...)` on each field to give the model field-level guidance, similar to parameter descriptions in tool schemas.

---

### When to Use Each

**Simple**
Use function calling when the model needs to fetch external data or trigger an action. Use structured output when you just need the model's reply in a machine-readable format. They solve different problems.

**Technical**
- **Function calling:** model needs information it does not have, or needs to trigger a side effect (search, DB lookup, calculation, API call)
- **Structured output / JSON mode:** model has enough information in context but the reply needs to be parsed by code (extraction, classification, scoring, generating config)
- **Structured output inside a tool call loop:** common in agentic systems where each step both takes an action and returns structured data about that action's result

---

### The instructor Library

**Simple**
Pydantic with `response_format` works when you are asking the model to produce structured output directly. The instructor library goes one step further: it wraps the API client so you can pass a Pydantic model directly, and if the model's output fails validation it automatically retries with the validation error in the prompt. This is useful for extracting structured data from free text - resumes, documents, user messages - where the model may not get the schema right on the first try.

**Technical**
instructor patches the OpenAI (and Anthropic) client via `instructor.from_openai(client)`. Internally it converts your Pydantic model to a tool schema, calls the API, validates the response against the model, and retries up to `max_retries` times if validation fails - passing the Pydantic error message back as a correction prompt. Output is a typed Python object, not a dict. Use instructor when: (1) extracting from messy free text, (2) you want automatic retries on schema mismatch, (3) you need the same extraction code to work across OpenAI and Anthropic without rewriting schemas.

# **Code Demos**

> **Note - Module 1 preview:** All demos below use the OpenAI Python SDK and Pydantic. These libraries are not introduced until Class 14 (LLM SDKs) in Module 2. In Module 1 the focus is on concepts - you can read through the demos now to understand the mechanics, but run the cells after Class 14 once your Python environment is set up.

### **Demo 1 - Define and Call a Tool (Weather Lookup)**

In [ ]:
# pip install openai
import json
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

# Step 1: Define the tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Returns temperature in Celsius and a condition string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'Bangalore'"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Step 2: First API call - model decides to use the tool
messages = [{"role": "user", "content": "What is the weather in Bangalore right now?"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools
)

first_message = response.choices[0].message
print("Model's response (raw):")
print(f"  finish_reason: {response.choices[0].finish_reason}")
print(f"  tool_calls: {first_message.tool_calls}")

# Step 3: Extract the tool call and simulate execution
tool_call = first_message.tool_calls[0]
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)
print(f"\nModel wants to call: {tool_name}({tool_args})")

# Simulate the real tool (in production this calls a real weather API)
def get_weather(city: str) -> dict:
    return {"city": city, "temperature_c": 28, "condition": "partly cloudy"}

tool_result = get_weather(**tool_args)
print(f"Tool returned: {tool_result}")

# Step 4: Feed result back and get final response
messages.append(first_message)  # append the assistant's tool call
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)
})

final_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools
)

print(f"\nFinal answer: {final_response.choices[0].message.content}")

**What to look at in the output:**
- The first response has `finish_reason: tool_calls` and no natural language content - the model stopped to request a tool, not to answer.
- The `tool_calls` object contains the exact function name and arguments the model chose, as valid JSON.
- The final answer uses the number returned by the tool - the model did not invent a temperature.

### **Demo 2 - The Tool Arbiter Pattern**

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

# Tool registry - maps names to callables
def search_database(query: str, limit: int) -> list:
    """Simulated DB search."""
    return [{"id": 1, "name": "Alice", "score": 92}, {"id": 2, "name": "Bob", "score": 87}][:limit]

TOOL_REGISTRY = {
    "search_database": search_database
}

# Arbiter: validates the model's tool call before execution
class ToolArbiter:
    MAX_LIMIT = 10
    ALLOWED_TOOLS = {"search_database"}

    def validate_and_dispatch(self, tool_call) -> dict:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        # Check 1: tool must be in the allowed list
        if name not in self.ALLOWED_TOOLS:
            return {"error": f"Tool '{name}' is not permitted."}

        # Check 2: argument types and bounds
        if name == "search_database":
            if not isinstance(args.get("query"), str) or not args["query"].strip():
                return {"error": "query must be a non-empty string"}
            limit = args.get("limit", 5)
            if not isinstance(limit, int) or limit < 1 or limit > self.MAX_LIMIT:
                return {"error": f"limit must be an integer between 1 and {self.MAX_LIMIT}"}
            args["limit"] = limit  # normalize

        # Validation passed - dispatch
        print(f"  [Arbiter] Approved: {name}({args})")
        result = TOOL_REGISTRY[name](**args)
        return {"result": result}


arbiter = ToolArbiter()

tools = [{
    "type": "function",
    "function": {
        "name": "search_database",
        "description": "Search student records by name or keyword. Returns matching students.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search term"},
                "limit": {"type": "integer", "description": "Max results to return (1-10)", "default": 5}
            },
            "required": ["query"]
        }
    }
}]

# Run the full loop with arbiter in the middle
messages = [{"role": "user", "content": "Find students named Alice in the database."}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools
)

msg = response.choices[0].message
if response.choices[0].finish_reason == "tool_calls":
    tool_call = msg.tool_calls[0]
    print(f"Model proposed: {tool_call.function.name}({tool_call.function.arguments})")

    # Arbiter validates and dispatches
    outcome = arbiter.validate_and_dispatch(tool_call)
    print(f"Arbiter outcome: {outcome}")

    # Feed result back
    messages.append(msg)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(outcome)
    })

    final = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
    print(f"\nFinal answer: {final.choices[0].message.content}")

# Test the arbiter's rejection path
print("\n--- Testing arbiter rejection ---")
class FakeBadToolCall:
    class function:
        name = "delete_all_records"
        arguments = '{}'
    id = "fake-id"

result = arbiter.validate_and_dispatch(FakeBadToolCall())
print(f"Arbiter blocked: {result}")

**What to look at in the output:**
- The `[Arbiter] Approved` line fires between the model's proposal and actual tool execution - this is exactly the separation of concerns the pattern enforces.
- The rejection test shows the arbiter blocking a tool the model was not allowed to call, with no exception and no execution.
- In production this block would log the attempt and could trigger an alert.

### **Demo 3 - Structured Output with Pydantic**

> **Note:** This demo uses Pydantic (covered in Class 14+). Run after completing Module 2 setup.

In [ ]:
# pip install openai pydantic
from pydantic import BaseModel, Field
from openai import OpenAI
import json

client = OpenAI()

# Define the output schema as a Pydantic model
class CandidateEvaluation(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate")
    technical_score: int = Field(description="Technical skill score from 0 to 10")
    communication_score: int = Field(description="Communication clarity score from 0 to 10")
    hire_recommendation: bool = Field(description="True if candidate should advance to next round")
    summary: str = Field(description="One sentence summary of the evaluation")

# Interview transcript to evaluate
transcript = """
Interviewer: Can you explain how a hash map works?
Candidate: Sure! A hash map stores key-value pairs. When you insert a key, it runs the key through a hash function that converts it to an index in an underlying array. Then the value is stored at that index. Lookup is O(1) average case. Collisions are handled by chaining or open addressing.
Interviewer: What is O(1) mean?
Candidate: It means constant time - the operation takes roughly the same amount of time regardless of how many items are in the map.
"""

# Method 1: JSON mode (model chooses keys)
response_json_mode = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Evaluate this interview transcript. Return JSON only."},
        {"role": "user", "content": transcript}
    ],
    response_format={"type": "json_object"}
)
raw_json = json.loads(response_json_mode.choices[0].message.content)
print("JSON mode output (model chooses schema):")
print(json.dumps(raw_json, indent=2))

print("\n" + "-"*50 + "\n")

# Method 2: Structured output with enforced schema (OpenAI Structured Outputs)
schema = CandidateEvaluation.model_json_schema()
response_structured = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Evaluate this interview transcript. The candidate's name is not given - use 'Unknown'."},
        {"role": "user", "content": transcript}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "CandidateEvaluation",
            "strict": True,
            "schema": schema
        }
    }
)

raw = json.loads(response_structured.choices[0].message.content)
evaluation = CandidateEvaluation(**raw)  # parse into typed object

print("Structured output (schema-enforced, Pydantic-parsed):")
print(f"  Technical score:     {evaluation.technical_score}/10")
print(f"  Communication score: {evaluation.communication_score}/10")
print(f"  Hire recommendation: {evaluation.hire_recommendation}")
print(f"  Summary: {evaluation.summary}")
print(f"\nType of evaluation object: {type(evaluation)}")

**What to look at in the output:**
- JSON mode output varies in key names each run - the model picks its own schema. Useful for quick extraction, not reliable for downstream code.
- Structured output always has exactly the fields defined in `CandidateEvaluation`, with correct types, every time.
- The final `type()` shows you have a real Python object, not a dict - you get `.technical_score` attribute access and Pydantic's validation on every field.

### **Demo 4 - Multi-Tool Loop (Calculator + Search)**

In [ ]:
import json
import math
from openai import OpenAI

client = OpenAI()

# Two tools: a calculator and a fake search
def calculator(expression: str) -> dict:
    """Safely evaluate a math expression."""
    try:
        # Restrict to safe operations only
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
        allowed['__builtins__'] = {}
        result = eval(expression, allowed)  # noqa: S307
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}

def search_knowledge(query: str) -> dict:
    """Simulated knowledge base search."""
    kb = {
        "compound interest formula": "A = P * (1 + r/n)^(n*t) where P=principal, r=annual rate, n=compounds per year, t=years",
        "default": "No result found for that query."
    }
    for key in kb:
        if key in query.lower():
            return {"result": kb[key]}
    return {"result": kb["default"]}

TOOL_REGISTRY = {"calculator": calculator, "search_knowledge": search_knowledge}

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression. Use Python math syntax. Example: '1000 * (1 + 0.08/12) ** (12*5)'",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "Math expression to evaluate"}},
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search a knowledge base for a concept or formula.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "What to look up"}},
                "required": ["query"]
            }
        }
    }
]

# Question that requires both tools: look up formula, then compute
messages = [{
    "role": "user",
    "content": "What is the compound interest formula? Then calculate: 50000 invested at 8% annual rate, compounded monthly, for 10 years."
}]

print("Starting multi-tool loop...\n")
max_steps = 5

for step in range(max_steps):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )
    msg = response.choices[0].message
    finish = response.choices[0].finish_reason

    print(f"Step {step + 1}: finish_reason = {finish}")

    if finish == "stop":
        print(f"\nFinal answer:\n{msg.content}")
        break

    if finish == "tool_calls":
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  Calling {tc.function.name}({args})")
            result = TOOL_REGISTRY[tc.function.name](**args)
            print(f"  Result: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result)
            })

**What to look at in the output:**
- Watch the step count - the model calls `search_knowledge` first, then `calculator`, then stops. Each step has a different `finish_reason`.
- The calculator result is exact floating-point arithmetic, not the model's approximation.
- The loop terminates naturally when `finish_reason == "stop"` - you do not need to count steps in simple cases, but `max_steps` is a production safeguard against infinite loops.

## Extension: instructor Library (Optional)

The `instructor` library wraps the OpenAI (and Anthropic) client to add automatic retries when the model's structured output fails schema validation. If you have covered Demos 1-4 and want to see how to extract structured data from messy free text with less boilerplate, run the demo below. Otherwise, come back here after Class 14 when you have seen structured outputs in a full project context.

### **Demo 5 - instructor Library: Structured Extraction from Free Text**

**What is `instructor`?** It is a Python library that wraps the OpenAI (and Anthropic) SDK to add one critical feature: if the model returns JSON that fails your Pydantic schema validation, `instructor` automatically retries the call with the validation error fed back to the model. This means you stop writing `try/except json.JSONDecodeError` loops and let the library handle it. Install with `pip install instructor`.

In [ ]:
# pip install instructor openai pydantic
import instructor
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Optional

# instructor patches the OpenAI client - one line change from standard usage
client = instructor.from_openai(OpenAI())

# Define the schema you want to extract
class JobPosting(BaseModel):
    role: str = Field(description="Job title")
    company: Optional[str] = Field(description="Company name if mentioned")
    required_skills: list[str] = Field(description="List of required technical skills")
    experience_years: Optional[int] = Field(description="Required years of experience")
    is_remote: bool = Field(description="True if the role is fully remote")

# Raw free text - no structure, no labels
free_text = """
We are looking for a Senior Backend Engineer at Razorpay to join our payments
infrastructure team. Must have 5+ years with Go or Python. Strong understanding
of distributed systems and Kafka required. PostgreSQL experience a plus.
This is a hybrid role, 3 days from our Bangalore office.
"""

# instructor handles schema conversion, API call, validation, and retries
posting = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=JobPosting,
    messages=[{"role": "user", "content": f"Extract job posting details:\n\n{free_text}"}]
)

print(f"Role:        {posting.role}")
print(f"Company:     {posting.company}")
print(f"Skills:      {posting.required_skills}")
print(f"Experience:  {posting.experience_years} years")
print(f"Remote:      {posting.is_remote}")
print(f"\nType: {type(posting)}")  # a real Python object, not a dict

**What to look at in the output:**
- `type(posting)` shows a real `JobPosting` Python object - attribute access (`.role`, `.required_skills`) not dict keys.
- The model extracted `is_remote: False` from "hybrid role, 3 days from office" - not an explicit label, a reasoned inference.
- If the model returned invalid JSON or missed a required field, instructor would automatically retry with the Pydantic error in the prompt. Demo 3 (Pydantic with `response_format`) does not retry - it raises an exception.
- Swap `instructor.from_openai` for `instructor.from_anthropic` and the same code works with Claude, same Pydantic schema, no rewriting.

# **Key Takeaways**

1. LLMs have three hard limits that prompting cannot fix: hallucinations (they guess when uncertain), stale knowledge (training cutoff), and no real arithmetic. Tools solve all three.

2. The model never executes code. It outputs a structured JSON request. Your application executes the tool and feeds the result back. The model is a planner, not a runner.

3. The Tool Arbiter pattern is mandatory in any production system. It sits between the model's proposal and actual execution - validates schema, checks authorization, logs every call, and can reject before any side effect occurs.

4. Tool schema descriptions are the primary signal the model uses to decide which tool to call and what arguments to provide. Narrow, precise descriptions produce reliable tool use. Vague descriptions produce errors.

5. Pre-validation (before dispatch) catches malformed model outputs. Post-validation (after tool return) catches unexpected tool results. Both are needed in production.

6. Structured outputs and function calling solve different problems. Function calling is for fetching external data or triggering actions. Structured outputs are for getting the model's own reply in a machine-readable format.

7. Pydantic models give you schema definition, JSON Schema generation, and runtime validation in one object. Use `Field(description=...)` on each field to give the model the same guidance as tool parameter descriptions.

# **Quick Check**

1. A user asks an LLM what the current USD/INR exchange rate is. The model confidently gives a number from its training data. Which of the three LLM problems is this, and which tool type fixes it?

2. After the model outputs a `tool_calls` response, what is the correct sequence of steps before the model produces its final reply? Write the steps in order.

3. What is the Tool Arbiter's job? At what point in the tool call loop does it run, and what are two categories of checks it performs?

4. You have a tool that deletes student records. Why is it dangerous to expose this to the model without an arbiter, even if you trust the model? What specific attack makes this a security issue?

5. What is the difference between `response_format: json_object` and `response_format: json_schema`? When would you use each?

6. You define a Pydantic model with five fields and send it as a structured output schema. The model returns valid JSON but omits one required field. What happens when you call `YourModel(**raw_json)`?

7. In Demo 4, the loop runs multiple steps. What finish_reason tells you the model is done and ready to give a final answer? What finish_reason tells you it wants to call another tool?

## **Quick Check - Answers**

---

*Attempt the questions above before reading these answers.*

---

**1.** This is the knowledge cutoff problem - the model's training data is static and does not include real-time exchange rates. The fix is a live data tool: a web search tool or a dedicated currency API tool that fetches current rates at query time.

**2.** (1) Receive the model's response containing `tool_calls`. (2) Extract the tool name and arguments from `tool_calls[0]`. (3) Execute the tool function locally with those arguments. (4) Append the tool result as a message with `role: "tool"` and the matching `tool_call_id`. (5) Send the updated conversation history back to the model. (6) Receive the model's final text response.

**3.** The Tool Arbiter runs between step 2 and step 3 of the loop - after the model requests a tool call, before the tool executes. It performs: (a) safety and scope checks (is this tool allowed, are arguments within expected ranges, could this be a destructive operation?); (b) authorization checks (does this user have permission to call this tool with these arguments?).

**4.** Even a trustworthy model can be manipulated via prompt injection - malicious text in a retrieved document could instruct the model to call the delete tool with a real student ID. The model is not "hacked" but it may follow the injected instruction. The arbiter adds a non-LLM verification layer that cannot be bypassed by text input.

**5.** `json_object` guarantees valid JSON but enforces no schema - the model chooses which fields to include. `json_schema` enforces a specific structure: required fields, types, and nesting must match the schema you define. Use `json_object` when structure varies; use `json_schema` (or Pydantic) when downstream code expects a fixed schema.

**6.** A `ValidationError` is raised by Pydantic immediately. It reports which required field is missing and refuses to create the model instance. You must handle this exception - either retry the API call or fall back gracefully.

**7.** `finish_reason = "stop"` (or `"end_turn"` in Anthropic) means the model is done and the message is the final answer. `finish_reason = "tool_calls"` means the model is requesting a tool - extract the call, execute it, and continue the loop.